# TP1 : Prétraitement des Données — Python (Google Colab) - Zakaria Ouadifi - GDNC 2
**Objectif :** Mettre en œuvre un pipeline complet de prétraitement des données médicales en Python, en couvrant : exploration, nettoyage, imputation des valeurs manquantes et traitement des outliers.

---

## I. Chargement et Exploration des Données

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Loading the file
file_path = '/content/drive/MyDrive/Google Colab/TP1-AI/namefile.xlsx'
df = pd.read_excel(file_path)

# 2. Display statistical description (equivalent to summary() in R)
print("--- Description Statistique ---")
display(df.describe(include='all'))

# 3. Display pairplot (equivalent to pairs() in R)
print("\n--- Nuage de points croisés (Pairs) ---")
sns.pairplot(df)
plt.show()



### 📝 Discussion et Analyse de la Description Statistique

Après l'exécution de `describe()` et `pairplot()`, on peut faire les observations suivantes :

**Variables numériques (age, tauxmax, depression) :**
- **age** : La moyenne est autour de 54 ans avec un écart-type modéré, ce qui indique une population adulte à risque cardiovasculaire. La distribution semble relativement symétrique.
- **tauxmax** : Variable représentant le taux cardiaque maximal atteint. Une valeur minimale anormalement basse (71.0) est visible — c'est une piste pour un outlier à investiguer.
- **depression** : Valeur moyenne basse avec une forte asymétrie vers 0, suggérant que beaucoup de patients n'ont pas de dépression ST significative.

**Variables catégorielles (typedouleur, sucre, angine) :**
- Ces variables ont un faible nombre de modalités uniques, ce qui est cohérent avec des indicateurs médicaux binaires ou à catégories limitées.
- Les valeurs manquantes restent faibles (< 2%), sauf pour `tension` (80%) et `fumeur` (58%) qui seront traitées en étape 4.

**Pairplot :**
- On observe une corrélation négative visible entre `age` et `tauxmax` : les patients plus âgés ont tendance à avoir un taux cardiaque maximal plus faible, ce qui est physiologiquement cohérent.
- La variable `depression` présente une dispersion large avec plusieurs valeurs proches de 0, confirmant son asymétrie.
- Pas de corrélation linéaire forte entre `age` et `depression`.

---
## II. Données Invariantes

Une variable invariante a une variance nulle : toutes ses valeurs sont identiques (ou il n'y a qu'une seule valeur unique). Elle n'apporte aucune information discriminante et doit être supprimée.

In [ ]:
# ==========================================
# Étape 2 : Données invariantes (Corrigée)
# ==========================================
print("--- Traitement des données invariantes ---")

# dropna=False permet de compter les cases vides (NaN) comme une valeur unique
colonnes_invariantes = df.columns[df.nunique(dropna=False) <= 1]
print(f"Colonnes réellement invariantes détectées : {list(colonnes_invariantes)}")

# Supprimer ces colonnes du DataFrame
df = df.drop(columns=colonnes_invariantes)
print("Dimensions après suppression des invariantes :", df.shape)

---
## III. Traitement des Doublons

Les doublons peuvent biaiser l'entraînement des modèles en surpondérant certaines observations. La méthode `drop_duplicates()` supprime toutes les lignes identiques.

In [ ]:
# ==========================================
# Étape 3 : Traiter les doublons
# ==========================================
print("\n--- Traitement des doublons ---")

# Compter le nombre de doublons avant de les supprimer
nb_doublons = df.duplicated().sum()
print(f"Nombre de lignes en double trouvées : {nb_doublons}")

# Supprimer les doublons (équivalent de DF <- DF[!duplicated(DF),] en R)
df = df.drop_duplicates()

print("Dimensions du DataFrame après suppression des doublons :", df.shape)
display(df.head())

---
## IV. Données Éparses (Sparse Data)

Une colonne est dite *éparse* si elle contient un taux trop élevé de valeurs manquantes. Au-delà de **40%** de NaN, l'imputation introduirait plus de bruit que d'information réelle.

In [ ]:
# ==========================================
# Étape 4 : Données éparses (> 40% de NaN)
# ==========================================
print("--- Traitement des données éparses ---")

# 1. Calculer le pourcentage de valeurs manquantes (NaN) par colonne
pourcentage_manquants = (df.isnull().sum() / len(df)) * 100

print("Pourcentage de données manquantes par colonne :")
# On n'affiche que celles qui ont des valeurs manquantes pour y voir plus clair
print(pourcentage_manquants[pourcentage_manquants > 0].round(2).astype(str) + ' %')

# 2. Identifier les colonnes avec PLUS de 40% de données manquantes
colonnes_eparses = pourcentage_manquants[pourcentage_manquants > 40].index.tolist()
print(f"\nColonnes éparses détectées (> 40% manquants) : {colonnes_eparses}")

# 3. Traitement : On supprime ces colonnes du DataFrame
df = df.drop(columns=colonnes_eparses)

print("\nDimensions du DataFrame après suppression des données éparses :", df.shape)
display(df.head())

### 📝 Discussion du Traitement des Données Éparses

Deux colonnes dépassent le seuil de 40% de NaN :
- **`tension`** : 80% de valeurs manquantes → suppression obligatoire. Imputer 80% des valeurs reviendrait à *inventer* la quasi-totalité de la colonne, ce qui biaiserait fortement l'analyse.
- **`fumeur`** : 58% de valeurs manquantes → même raisonnement. Plus de la moitié des données sont absentes.

**Pourquoi supprimer plutôt que compléter ?**
Le seuil de 40% est une règle empirique largement acceptée. En dessous de ce seuil, des méthodes d'imputation (kNN, MICE) peuvent estimer les valeurs manquantes de façon fiable car elles s'appuient sur suffisamment de données réelles. Au-dessus, toute imputation devient trop spéculative.

Les colonnes restantes avec des taux < 2% de NaN (`typedouleur`, `sucre`, `tauxmax`, `angine`, `depression`) seront traitées par imputation dans l'étape suivante.

---
## V. Imputation des Valeurs Manquantes

Après nettoyage, il reste **19 NaN** répartis sur 5 colonnes. Trois méthodes d'imputation sont comparées, puis une méthode hybride est proposée, et enfin MICE est testée comme méthode supplémentaire.

In [ ]:
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer  # Requis pour activer IterativeImputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

In [ ]:
print("--- Étape 5 : Imputation des valeurs manquantes ---")

# =========================================================
# 0. Préparation : Conversion du texte en nombres
# =========================================================
df_num = df.copy()
colonnes_texte = df_num.select_dtypes(include=['object']).columns

# On transforme les catégories de texte en codes numériques
for col in colonnes_texte:
    df_num[col] = df_num[col].astype('category').cat.codes
    # L'encodage met les cases vides (NaN) à -1.
    # On les remet à NaN pour que l'algorithme comprenne qu'elles sont manquantes.
    df_num[col] = df_num[col].replace(-1, np.nan)

In [ ]:
# =========================================================
# 1. Méthode kNN (k=5)
# =========================================================
imputer_knn = KNNImputer(n_neighbors=5)
don_knn_array = imputer_knn.fit_transform(df_num)
don_knn = pd.DataFrame(don_knn_array, columns=df.columns)
print("✅ Imputation kNN terminée.")

In [ ]:
# =========================================================
# 2. Méthode missForest (Random Forest)
# =========================================================
# L'équivalent de missForest en Python est l'IterativeImputer avec un modèle Random Forest
rf_estimator = RandomForestRegressor(n_estimators=50, random_state=42)
imputer_rf = IterativeImputer(estimator=rf_estimator, max_iter=10, random_state=42)
don_missForest_array = imputer_rf.fit_transform(df_num)
don_missForest = pd.DataFrame(don_missForest_array, columns=df.columns)
print("✅ Imputation missForest terminée.")

In [ ]:
# =========================================================
# 3. Méthode LOCF (Last Observation Carried Forward)
# =========================================================
# LOCF est la méthode la plus simple, on recopie la valeur de la ligne précédente (ffill)
# On utilise df original ici, car LOCF n'a pas besoin de nombres, il marche très bien avec du texte
don_locf = df.copy()
don_locf = don_locf.ffill().bfill() # bfill() sécurise le cas où la toute 1ère ligne de la colonne est vide
print("✅ Imputation LOCF terminée.")

In [ ]:
# =========================================================
# Vérification des résultats
# =========================================================
print("\n--- Vérification du nombre de valeurs manquantes (NaN) restantes ---")
print(f"Dans le jeu de données d'origine : {df.isnull().sum().sum()} NaN")
print(f"Dans don_knn                     : {don_knn.isnull().sum().sum()} NaN")
print(f"Dans don_missForest              : {don_missForest.isnull().sum().sum()} NaN")
print(f"Dans don_locf                    : {don_locf.isnull().sum().sum()} NaN")

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error

print("--- Étape 6 : Comparaison des erreurs d'imputation ---")

def evaluer_imputation(df_original):
    # 1. On isole un jeu de données 100% sain (sans aucun vrai NaN)
    df_propre = df_original.dropna().copy()

    # Pour simplifier le calcul d'erreur mathématique, on teste sur les colonnes numériques
    colonnes_test = ['tauxmax', 'depression']
    df_test = df_propre[colonnes_test].copy()

    # 2. On sabote nos données : on crée 10% de trous (NaN) aléatoires
    np.random.seed(42) # Seed pour que l'aléatoire soit identique à chaque test
    df_troue = df_test.copy()
    masque_trous = np.random.rand(*df_troue.shape) < 0.10
    df_troue[masque_trous] = np.nan

    # 3. Déploiement des algorithmes pour réparer les dégâts
    # kNN
    imputer_knn = KNNImputer(n_neighbors=5)
    knn_repari = imputer_knn.fit_transform(df_troue)

    # missForest (RandomForest)
    imputer_rf = IterativeImputer(estimator=RandomForestRegressor(n_estimators=50, random_state=42), random_state=42)
    rf_repari = imputer_rf.fit_transform(df_troue)

    # LOCF
    locf_repari = df_troue.ffill().bfill().values

    # 4. Calcul de l'erreur (RMSE - Root Mean Squared Error) uniquement sur les cases trouées
    # Plus le score est bas, plus la prédiction était proche de la réalité.
    vraies_valeurs = df_test.values[masque_trous]

    erreur_knn = np.sqrt(mean_squared_error(vraies_valeurs, knn_repari[masque_trous]))
    erreur_rf = np.sqrt(mean_squared_error(vraies_valeurs, rf_repari[masque_trous]))
    erreur_locf = np.sqrt(mean_squared_error(vraies_valeurs, locf_repari[masque_trous]))

    return erreur_knn, erreur_rf, erreur_locf

# Exécution de notre environnement de test
err_knn, err_rf, err_locf = evaluer_imputation(df)

print(f"Marge d'erreur kNN         : {err_knn:.2f}")
print(f"Marge d'erreur missForest  : {err_rf:.2f}")
print(f"Marge d'erreur LOCF        : {err_locf:.2f}")

# Le verdict
erreurs = {'kNN': err_knn, 'missForest': err_rf, 'LOCF': err_locf}
meilleur = min(erreurs, key=erreurs.get)
print(f"\n🏆 L'algorithme le plus précis est : {meilleur} (car son taux d'erreur est le plus bas)")

### Q5 — Méthode Hybride

**Idée :** Combiner la meilleure méthode pour les variables numériques (kNN) avec la méthode la plus adaptée aux variables catégorielles (LOCF/Mode), car les algorithmes mathématiques comme kNN sont moins naturels sur des valeurs discrètes non ordinales.

In [ ]:
# =========================================================
# Méthode Hybride : kNN pour numériques, Mode pour catégorielles
# =========================================================
print("--- Méthode Hybride : kNN (numériques) + Mode (catégorielles) ---")

don_hybride = df.copy()

# Identifier les colonnes numériques et catégorielles
cols_numeriques  = don_hybride.select_dtypes(include=['number']).columns.tolist()
cols_categorielles = don_hybride.select_dtypes(include=['object']).columns.tolist()

print(f"Colonnes numériques  : {cols_numeriques}")
print(f"Colonnes catégorielles : {cols_categorielles}")

# 1. Pour les colonnes NUMÉRIQUES → kNN imputation
if cols_numeriques:
    imputer_hybride = KNNImputer(n_neighbors=5)
    don_hybride[cols_numeriques] = imputer_hybride.fit_transform(don_hybride[cols_numeriques])
    print("✅ kNN appliqué sur les colonnes numériques.")

# 2. Pour les colonnes CATÉGORIELLES → imputation par le MODE (valeur la plus fréquente)
for col in cols_categorielles:
    if don_hybride[col].isnull().sum() > 0:
        mode_val = don_hybride[col].mode()[0]
        don_hybride[col] = don_hybride[col].fillna(mode_val)
        print(f"  Mode appliqué sur '{col}' → valeur imputée : '{mode_val}'")

# Vérification
nan_restants = don_hybride.isnull().sum().sum()
print(f"\n✅ Imputation hybride terminée. NaN restants : {nan_restants}")
display(don_hybride.head())

### Q6 — Autre Méthode d'Imputation : MICE (via IterativeImputer avec BayesianRidge)

**MICE** (Multivariate Imputation by Chained Equations) est une méthode d'imputation multiple. Pour chaque colonne contenant des NaN, elle entraîne un modèle de régression en utilisant toutes les autres colonnes comme prédicteurs, puis itère jusqu'à convergence.

En Python, l'`IterativeImputer` de scikit-learn est l'équivalent direct de MICE. Par défaut, il utilise un régresseur **BayesianRidge**, qui est plus adapté que Random Forest pour des données continues de petite taille.

In [ ]:
# =========================================================
# Méthode MICE (Multivariate Imputation by Chained Equations)
# =========================================================
from sklearn.linear_model import BayesianRidge

print("--- Méthode MICE (IterativeImputer + BayesianRidge) ---")

# MICE utilise IterativeImputer avec BayesianRidge comme estimateur par défaut
imputer_mice = IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=42)
don_mice_array = imputer_mice.fit_transform(df_num)
don_mice = pd.DataFrame(don_mice_array, columns=df.columns)

print(f"✅ Imputation MICE terminée. NaN restants : {don_mice.isnull().sum().sum()}")

# Comparaison RMSE incluant MICE
print("\n--- Comparaison étendue des erreurs d'imputation (avec MICE) ---")

def evaluer_avec_mice(df_original):
    df_propre = df_original.dropna().copy()
    colonnes_test = ['tauxmax', 'depression']
    df_test = df_propre[colonnes_test].copy()

    np.random.seed(42)
    df_troue = df_test.copy()
    masque_trous = np.random.rand(*df_troue.shape) < 0.10
    df_troue[masque_trous] = np.nan

    # kNN
    knn_r = KNNImputer(n_neighbors=5).fit_transform(df_troue)
    # missForest
    rf_r = IterativeImputer(estimator=RandomForestRegressor(n_estimators=50, random_state=42), random_state=42).fit_transform(df_troue)
    # LOCF
    locf_r = df_troue.ffill().bfill().values
    # MICE
    mice_r = IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=42).fit_transform(df_troue)

    vraies = df_test.values[masque_trous]
    from sklearn.metrics import mean_squared_error
    rmse = lambda pred: np.sqrt(mean_squared_error(vraies, pred[masque_trous]))

    return {'kNN': rmse(knn_r), 'missForest': rmse(rf_r), 'LOCF': rmse(locf_r), 'MICE': rmse(mice_r)}

resultats = evaluer_avec_mice(df)
for methode, erreur in sorted(resultats.items(), key=lambda x: x[1]):
    print(f"  {methode:15s} → RMSE : {erreur:.2f}")

meilleur = min(resultats, key=resultats.get)
print(f"\n🏆 Meilleure méthode : {meilleur}")

# Visualisation comparative
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 5))
bars = plt.bar(resultats.keys(), resultats.values(),
               color=['#2196F3', '#4CAF50', '#FF9800', '#9C27B0'], edgecolor='white', linewidth=1.5)
plt.title("Comparaison des erreurs RMSE d'imputation (4 méthodes)", fontsize=13, fontweight='bold')
plt.xlabel("Méthode d'imputation")
plt.ylabel("RMSE (plus bas = meilleur)")
plt.grid(axis='y', linestyle='--', alpha=0.5)
for bar, val in zip(bars, resultats.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.2f}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

---
## VI. Données Atypiques (Outliers)

### Q1 — Types de Valeurs Aberrantes

Les valeurs aberrantes se classent en deux catégories :

- **Univariées** : anomalie visible dans la distribution d'une seule variable. Exemple : un `tauxmax` de 71 bpm quand la moyenne est autour de 150 bpm. Détectable avec un boxplot ou la règle IQR.

- **Multivariées** : la valeur n'est pas aberrante dans chaque dimension prise isolément, mais la *combinaison* de valeurs est incohérente. Exemple : un patient très jeune (25 ans) avec un taux de depression ST très élevé. Détectable avec des méthodes comme LOF (Local Outlier Factor) ou Mahalanobis.

Dans ce TP, nous traitons uniquement les outliers **univariés** sur les variables `age` et `tauxmax`.

### Q2 — Causes des Valeurs Aberrantes

Les causes peuvent être classées en deux grandes catégories :

**Naturelles (données réelles) :**
- Variabilité biologique extrême (ex : patient avec une condition médicale rare)
- Données correctes mais rares dans la population

**Artificielles (erreurs) :**
- *Erreur de saisie* : faute de frappe lors de l'encodage (ex : 710 saisi à la place de 71)
- *Erreur de mesure* : capteur défaillant ou mal étalonné
- *Erreur expérimentale* : protocole non respecté lors de la collecte
- *Erreur de traitement* : transformation incorrecte lors d'une conversion d'unité
- *Erreur d'échantillonnage* : individu hors population cible inclus par erreur

Dans notre cas, la valeur `tauxmax = 71` pour le patient 101 est suspectée d'être une **erreur de mesure ou de saisie**, car un tel taux cardiaque maximal lors d'un test d'effort est cliniquement incompatible avec un patient mobile.

### Q3 — Impact des Valeurs Aberrantes sur un Jeu de Données

Les outliers peuvent avoir des effets néfastes significatifs sur l'analyse :

- **Biais des statistiques descriptives** : la moyenne et l'écart-type sont très sensibles aux valeurs extrêmes (contrairement à la médiane).
- **Réduction de la puissance des tests statistiques** : augmentation de la variance d'erreur.
- **Perturbation de la normalité** : les outliers peuvent rompre l'hypothèse de normalité requise par certains tests.
- **Impact sur les modèles prédictifs** : en régression linéaire, un outlier peut fortement modifier la pente estimée. En clustering (k-means), il peut créer des centroïdes artificiels.
- **Influence sur les hypothèses de base** : les modèles supposant une distribution gaussienne seront faussés.

### Q4 — Méthodes de Détection des Valeurs Aberrantes

**Méthodes de visualisation :**
- **Boîte à moustaches (Boxplot)** : identifie les valeurs en dehors de Q1 - 1.5×IQR et Q3 + 1.5×IQR
- **Histogramme** : révèle les distributions asymétriques ou les valeurs isolées
- **Nuage de points (scatter plot)** : utile pour les outliers multivariés

**Méthodes statistiques :**
- **Règle IQR** (méthode utilisée dans ce TP) : robuste et non-paramétrique
- **Score Z** : valeur aberrante si |z| > 3 (suppose une distribution normale)
- **Distance de Mahalanobis** : pour les outliers multivariés

**Méthodes basées sur la densité :**
- **LOF (Local Outlier Factor)** : compare la densité locale d'un point à celle de ses voisins — efficace pour les structures non-globulaires
- **Isolation Forest** : isole les anomalies en construisant des arbres aléatoires

**Données catégorielles :**
- **AVF (Attribute Value Frequency)** : détecte les modalités rares comme aberrantes

### Q5 — Méthodes de Traitement des Outliers

Une fois détectés, les outliers peuvent être traités de plusieurs façons :

- **Suppression de l'observation** : simple mais entraîne une perte d'information. À utiliser si l'outlier est clairement une erreur et si le dataset est grand.
- **Transformation** : appliquer log(), sqrt() ou winsorisation (plafonner les valeurs aux percentiles 1% et 99%) pour réduire l'effet des extrêmes.
- **Traitement comme groupe distinct** : conserver les outliers mais les traiter séparément dans l'analyse (ex : sous-groupe clinique particulier).
- **Imputation** : remplacer la valeur aberrante par une estimation (kNN, médiane, médiane conditionnelle) — c'est la méthode retenue dans ce TP.
- **Maintien si outlier naturel** : si l'outlier est une vraie observation rare mais valide, il doit être conservé.

### Visualisation — Boîtes à Moustaches (Boxplot) par Variable

In [ ]:
# =========================================================
# Boxplots individuels pour 'age' et 'tauxmax' (équivalent de boxplot() en R)
# =========================================================
print("--- Visualisation des Boxplots : age et tauxmax ---")

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Boxplot pour 'age'
axes[0].boxplot(df['age'].dropna(), patch_artist=True,
                boxprops=dict(facecolor='#AED6F1', color='#2980B9'),
                medianprops=dict(color='red', linewidth=2),
                whiskerprops=dict(color='#2980B9'),
                capprops=dict(color='#2980B9'),
                flierprops=dict(marker='o', color='red', markersize=8))
axes[0].set_title("Boxplot — Variable 'age'", fontweight='bold')
axes[0].set_ylabel("Age (années)")
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

# Calcul des stats IQR pour 'age'
q1_age, q3_age = df['age'].quantile([0.25, 0.75])
iqr_age = q3_age - q1_age
out_age = df['age'][(df['age'] < q1_age - 1.5*iqr_age) | (df['age'] > q3_age + 1.5*iqr_age)]
print(f"Outliers détectés pour 'age' : {out_age.values.tolist()}")

# Boxplot pour 'tauxmax'
axes[1].boxplot(df['tauxmax'].dropna(), patch_artist=True,
                boxprops=dict(facecolor='#A9DFBF', color='#1E8449'),
                medianprops=dict(color='red', linewidth=2),
                whiskerprops=dict(color='#1E8449'),
                capprops=dict(color='#1E8449'),
                flierprops=dict(marker='o', color='red', markersize=8))
axes[1].set_title("Boxplot — Variable 'tauxmax'", fontweight='bold')
axes[1].set_ylabel("Taux max (bpm)")
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

# Calcul des stats IQR pour 'tauxmax'
q1_t, q3_t = df['tauxmax'].quantile([0.25, 0.75])
iqr_t = q3_t - q1_t
out_t = df['tauxmax'][(df['tauxmax'] < q1_t - 1.5*iqr_t) | (df['tauxmax'] > q3_t + 1.5*iqr_t)]
print(f"Outliers détectés pour 'tauxmax' : {out_t.values.tolist()} (indices : {out_t.index.tolist()})")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

print("--- Étape 7 : Détection et Visualisation des Outliers ---")

# 1. Fonction pour trouver les indices des anomalies (Méthode de la boîte à moustaches / IQR)
def trouver_outliers(colonne):
    Q1 = colonne.quantile(0.25)
    Q3 = colonne.quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    # Retourne les indices (numéros de ligne) des valeurs en dehors des limites
    return colonne[(colonne < lim_inf) | (colonne > lim_sup)].index.tolist()

# 2. Application sur les colonnes 'age' et 'tauxmax'
outliers_age = trouver_outliers(df['age'])
outliers_tauxmax = trouver_outliers(df['tauxmax'])

print(f"Indices des anomalies pour 'age'     : {outliers_age}")
print(f"Indices des anomalies pour 'tauxmax' : {outliers_tauxmax}")

# 3. Calcul de l'Intersection (anormal en âge ET en tauxmax) et de l'Union (anormal en âge OU en tauxmax)
outliers_inter = list(set(outliers_age) & set(outliers_tauxmax)) # Équivalent de intersect()
outliers_union = list(set(outliers_age) | set(outliers_tauxmax)) # Équivalent de union()

print(f"Anomalies Intersection (âge ET tauxmax) : {outliers_inter}")
print(f"Anomalies Union (âge OU tauxmax)        : {outliers_union}")

# 4. Visualisation (Équivalent de la boucle for avec 'points' en R)
plt.figure(figsize=(10, 6))

# On dessine d'abord tous les points "normaux" en bleu
plt.scatter(df['age'], df['tauxmax'], c='blue', label='Données normales', alpha=0.5)

# On superpose les outliers (ceux de l'Union) en rouge avec une grosse étoile
plt.scatter(df.loc[outliers_union, 'age'], df.loc[outliers_union, 'tauxmax'],
            c='red', marker='*', s=200, label='Données aberrantes (Outliers)')

plt.xlabel('Age')
plt.ylabel('Taux Max')
plt.title('Nuage de points : Age vs Taux Max avec anomalies')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
print("--- Étape finale : Correction de l'Outlier avec kNN ---")

# 1. On affiche la valeur aberrante avant correction
valeur_aberrante = df.loc[101, 'tauxmax']
print(f"Valeur aberrante du patient 101 avant correction : {valeur_aberrante}")

# 2. On efface cette erreur (on la remplace par NaN)
df_corrige = df.copy()
df_corrige.loc[outliers_union, 'tauxmax'] = np.nan

# 3. Préparation pour kNN (on doit re-transformer le texte en nombres temporairement)
df_num_corrige = df_corrige.copy()
for col in colonnes_texte:
    df_num_corrige[col] = df_num_corrige[col].astype('category').cat.codes
    df_num_corrige[col] = df_num_corrige[col].replace(-1, np.nan)

# 4. On utilise kNN pour deviner la vraie valeur
imputer_knn_final = KNNImputer(n_neighbors=5)
donnees_reparees = imputer_knn_final.fit_transform(df_num_corrige)

# 5. On remet la valeur réparée dans notre DataFrame propre
df_corrige['tauxmax'] = donnees_reparees[:, df.columns.get_loc('tauxmax')]

# Affichage du résultat !
nouvelle_valeur = df_corrige.loc[101, 'tauxmax']
print(f"Nouvelle valeur estimée par kNN pour le patient 101 : {nouvelle_valeur:.2f}")

---
## Conclusion

Ce TP a permis de mettre en oeuvre un pipeline complet de prétraitement des données médicales en Python. Voici un résumé des actions effectuées et des résultats :

| Étape | Action | Résultat |
|---|---|---|
| I | Exploration statistique + pairplot | Distribution identifiée, corrélations visualisées |
| II | Suppression colonne invariante | `nationaite` supprimée → (271, 10) |
| III | Suppression doublons | 1 doublon supprimé → (270, 10) |
| IV | Suppression données éparses | `tension` (80%) et `fumeur` (58%) supprimées → (270, 8) |
| V | Imputation kNN, missForest, LOCF | 19 NaN traités → 0 NaN restants |
| V | Évaluation RMSE | kNN meilleur (19.78), MICE compétitif |
| V | Méthode hybride | kNN (numérique) + Mode (catégoriel) |
| VI | Détection outliers IQR + boxplots | 1 outlier détecté : patient 101 (tauxmax=71) |
| VI | Correction outlier par kNN | tauxmax corrigé à 135.0 bpm |

**Conclusion principale :** La méthode kNN offre le meilleur compromis précision/simplicité pour ce jeu de données médical. La méthode hybride est recommandée en pratique pour gérer correctement les variables mixtes (numériques + catégorielles).